# Model 1 — Analizci (Metadata)
Türkçe metin → `{emotion, energy, bpm, key, instruments, vocal_style, suno_style_prompt}`

Başlamadan önce:
1. Sağ panel → **Add Data** → **Upload** → `dataset.jsonl`
2. Accelerator: **GPU T4 x2**
3. Hücreleri sırayla çalıştır

In [ ]:
# ── 1. Kurulum ────────────────────────────────────────────────────────────────
!pip install -q transformers datasets accelerate sentencepiece protobuf
print('Kurulum tamamlandi.')

In [ ]:
# ── 2. Dataset: Yükle + Metadata Hedefine Dönüştür ───────────────────────────
import glob, json
from datasets import Dataset

candidates = glob.glob('/kaggle/input/**/dataset.jsonl', recursive=True)
RAW_PATH = candidates[0] if candidates else '/kaggle/working/dataset.jsonl'
print(f'Ham dataset: {RAW_PATH}')

def fmt_instruments(v):
    return ', '.join(v) if isinstance(v, list) else str(v)

records = []
with open(RAW_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        d = json.loads(line)
        if 'input_text' not in d:
            continue
        # structured_lyrics çıkarıldı — hedef kısa ve net
        meta_target = {
            'emotion':          d.get('emotion', ''),
            'energy':           d.get('energy', 5),
            'bpm':              d.get('bpm', 90),
            'key':              d.get('key', 'A minor'),
            'instruments':      d.get('instruments', []),
            'vocal_style':      d.get('vocal_style', ''),
            'suno_style_prompt': d.get('suno_style_prompt', ''),
        }
        records.append({
            'input_text':  d['input_text'],
            'target_text': json.dumps(meta_target, ensure_ascii=False),
        })

ds = Dataset.from_list(records)
split = ds.train_test_split(test_size=0.10, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'Kayit: {len(records)} | Egitim: {len(train_ds)} | Dogrulama: {len(eval_ds)}')
print('Ornek hedef:', records[0]['target_text'])

In [ ]:
# ── 3. Konfigürasyon ──────────────────────────────────────────────────────────
BASE_MODEL     = 'google/mt5-small'   # flan-t5-small Türkçe bilmiyordu
OUTPUT_DIR     = '/kaggle/working/story-to-music-analyzer'
CHECKPOINT_DIR = '/kaggle/working/checkpoints-analyzer'

MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 128

EPOCHS         = 20
BATCH_SIZE     = 8
GRAD_ACCUM     = 2
LR             = 3e-4
WARMUP_RATIO   = 0.10
WEIGHT_DECAY   = 0.01
PATIENCE       = 4

print('Konfigurasyon tamam.')

In [ ]:
# ── 4. Model ve Tokenizer ─────────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Cihaz: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
print(f'Parametre sayisi: {model.num_parameters():,}')

In [ ]:
# ── 5. Tokenizasyon ───────────────────────────────────────────────────────────
from transformers import DataCollatorForSeq2Seq

def tokenize(batch):
    model_inputs = tokenizer(
        batch['input_text'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=batch['target_text'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in ids]
        for ids in labels['input_ids']
    ]
    return model_inputs

train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(tokenize,  batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8
)
print('Tokenizasyon tamam.')

In [ ]:
# ── 6. Eğitim ─────────────────────────────────────────────────────────────────
from pathlib import Path
import transformers
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback

total_steps  = (len(train_tok) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=3,
    fp16=(device == 'cuda'),
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    logging_steps=20,
    report_to='none',
    seed=42,
)

trainer_kwargs = dict(
    model=model, args=args,
    train_dataset=train_tok, eval_dataset=eval_tok,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)
ver = tuple(int(x) for x in transformers.__version__.split('.')[:2])
trainer_kwargs['processing_class' if ver >= (4, 46) else 'tokenizer'] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

checkpoints = sorted(Path(CHECKPOINT_DIR).glob('checkpoint-*'), key=lambda x: int(x.name.split('-')[-1])) if Path(CHECKPOINT_DIR).exists() else []
last_ckpt = str(checkpoints[-1]) if checkpoints else None
if last_ckpt:
    print(f'Checkpoint bulundu: {last_ckpt}')

trainer.train(resume_from_checkpoint=last_ckpt)

In [ ]:
# ── 7. Modeli Kaydet ──────────────────────────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

total_mb = sum(f.stat().st_size for f in Path(OUTPUT_DIR).rglob('*') if f.is_file()) / (1024**2)
print(f'Model kaydedildi: {OUTPUT_DIR}  ({total_mb:.1f} MB)')

In [ ]:
# ── 8. Smoke Test ─────────────────────────────────────────────────────────────
import textwrap

ornek = eval_ds[0]['input_text']
print('GIRDI:', textwrap.shorten(ornek, 150))

model.eval()
inputs = tokenizer(ornek, return_tensors='pt', max_length=MAX_INPUT_LEN, truncation=True).to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=4, early_stopping=True)

decoded = tokenizer.decode(out[0], skip_special_tokens=True)
print('\nCIKTI:', decoded)

# JSON parse kontrolü
try:
    parsed = json.loads(decoded)
    print('\nJSON gecerli. Alanlar:', list(parsed.keys()))
except json.JSONDecodeError as e:
    print('\nJSON HATASI:', e)

In [ ]:
# ── 9. Değerlendirme ──────────────────────────────────────────────────────────
import random

REQUIRED = {'emotion','energy','bpm','key','instruments','vocal_style','suno_style_prompt'}
random.seed(0)
samples = random.sample(range(len(eval_ds)), min(50, len(eval_ds)))
valid_json, coverages = 0, []

for idx in samples:
    inp = eval_ds[idx]['input_text']
    inputs = tokenizer(inp, return_tensors='pt', max_length=MAX_INPUT_LEN, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=4, early_stopping=True)
    gen = tokenizer.decode(out[0], skip_special_tokens=True)
    try:
        parsed = json.loads(gen)
        valid_json += 1
        coverages.append(len(REQUIRED & set(parsed.keys())) / len(REQUIRED))
    except:
        coverages.append(0.0)

n = len(samples)
print(f'Gecerli JSON : {valid_json}/{n} ({valid_json/n:.0%})')
print(f'Alan kapsami: {sum(coverages)/n:.0%}')

if valid_json/n >= 0.80 and sum(coverages)/n >= 0.85:
    print('\nModel 1 hazir. Simdi Model 2 (lyrics) egitebilirsin.')
else:
    print('\nHenuz yeterli degil — egitim loss grafigine bak, epoch artir.')

In [ ]:
# ── 10. İndir ─────────────────────────────────────────────────────────────────
import shutil
shutil.make_archive('/kaggle/working/story-to-music-analyzer', 'zip', OUTPUT_DIR)
print('Output panelinden story-to-music-analyzer.zip indirebilirsin.')